### Experiment with gpt 4o

In [ ]:
import pandas as pd
import libs.prompts as prompts
from libs.exp_utils import classify_dataset, evaluate_experiment
from libs.openai import call_openai
from tqdm import tqdm
import json
import os

In [ ]:
# Define model name
MODEL_NAME = "gpt-4o"

### Dataset load

In [3]:
# Load ground truth bug locations
with open("../../Datasets/bugLocationDappScan.json", "r") as f:
    bug_locations = json.load(f)

In [4]:
def normalize_path(path):
    if isinstance(path, str):
        return path.replace('\\', '/').replace('./', '').replace('.\\', '')
    return path

bugloc_dict = {
    normalize_path(item['file']): normalize_path(item['location'])
    for item in bug_locations
}


# Load the dataset and convert to appropriate data types
csv_path = "../../Datasets/dw.csv"
df = pd.read_csv(csv_path)
df['has_error'] = df['has_error'].astype(bool)
subset = df.reset_index(drop=True)
#subset.head()

#### Zero-shot Prompting

In [5]:
results_df = classify_dataset(subset, prompts.create_zeroshot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

400it [08:16,  1.24s/it]

Precision: 60.12
Recall: 99.50
F1-Score: 74.95
Confusion Matrix:
[[ 68 132]
 [  1 199]]


,predicted_has_error,actual_has_error,bug_type,location
0,1,0,Unprotected fallback function allowing unautho...,Lines 56-59
1,1,0,Denial of Service via Out-of-Gas (infinite loop),56-59
2,1,0,Constructor inheritance order causing multiple...,Lines 111-113 (contract DisableBad1)
3,1,0,Incorrect bitmask for boolean flag manipulation,22-28
4,1,0,Function Overloading Collision / Compilation E...,"Lines 11-13, 17-19, 25-27, 33-35, 41-43, 49-51"


In [6]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

,predicted_has_error,actual_has_error,bug_type,location,file_path,file_name
200,1,1,Transaction Order Dependence (Race Condition) ...,53,DAppSCAN-main/DAppSCAN-source/contracts/QuillA...,ERC20Permit.sol
201,1,1,SWC-103-Floating Pragma,3,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
202,1,1,SWC-103-Floating Pragma,4,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
203,1,1,Integer Overflow/Underflow and Unsafe Math Ope...,"Lines 89-91, 99-108",DAppSCAN-main/DAppSCAN-source/contracts/openze...,HackerGold.sol
204,1,1,Integer Underflow,122,DAppSCAN-main/DAppSCAN-source/contracts/PeckSh...,UserConfiguration.sol


In [7]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path])
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [8]:
# Roda a checagem
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    llm_categories.append(row['location'])  # Assuming 'bug_type' is the LLM's prediction
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")

    print("LOCATION: LINE - ", row['location'])
#print("LLM Categories:", llm_categories, "GT Categories:", gt_categories)



--- Rodando para idx 200 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/QuillAudits-1inch-token/1inch-token-99fd056f91005ca521a02a005f7bcd8f77e06afc/contracts/ERC20Permit.sol
LLM: Transaction Order Dependence (Race Condition) in permit function
GT: ['SWC-114-Transaction Order Dependence']

LOCATION: LINE -  53

--- Rodando para idx 201 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-SWAPP Protocol-project1/openzeppelin-contracts-3.3.0/contracts/access/AccessControl.sol
LLM: SWC-103-Floating Pragma
GT: ['SWC-103-Floating Pragma']

LOCATION: LINE -  3

--- Rodando para idx 202 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-GSPI Club-project2/openzeppelin-contracts-3.2.0/contracts/access/AccessControl.sol
LLM: SWC-103-Floating Pragma
GT: ['SWC-103-Floating Pragma', 'SWC-102-Outdated Compiler Version']

LOCATION: LINE -  4

--- Rodando para idx 203 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/openzeppelin-EtherCamp_Hacker_Gold/virtual-accelerator-252

In [9]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")


100%|██████████| 199/199 [03:35<00:00,  1.08s/it]

Precision: 100.00
Recall: 70.85
F1-Score: 82.94
Confusion Matrix:
[[  0   0]
 [ 58 141]]


#### Zero-shot CoT Prompting

In [5]:
results_df = classify_dataset(subset, prompts.create_zeroshot_cot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

400it [39:19,  5.90s/it]

Precision: 60.06
Recall: 97.00
F1-Score: 74.19
Confusion Matrix:
[[ 71 129]
 [  6 194]]


,predicted_has_error,actual_has_error,bug_type,location
0,1,0,Unprotected fallback function leading to poten...,Lines 56-59
1,0,0,NONE,0
2,1,0,Constructor Initialization Conflict / Inherita...,"Lines 111-113, 115-117, 120-123, 126-128"
3,1,0,Incorrect bitmask calculation,19-25
4,1,0,Function Overloading Collision,"Lines 13-16, 22-25, 32-35, 41-44, 50-53, 59-62"


In [6]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

,predicted_has_error,actual_has_error,bug_type,location,file_path,file_name
200,1,1,Transaction Order Dependence (Race Condition),53,DAppSCAN-main/DAppSCAN-source/contracts/QuillA...,ERC20Permit.sol
202,1,1,Floating Pragma / Outdated Compiler Version,4,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
203,1,1,Integer Overflow/Underflow and Deprecated Thro...,"Lines 30, 90",DAppSCAN-main/DAppSCAN-source/contracts/openze...,HackerGold.sol
204,1,1,Integer Underflow,Line 122,DAppSCAN-main/DAppSCAN-source/contracts/PeckSh...,UserConfiguration.sol
205,1,1,SWC-131-Presence of unused variables,213-222,DAppSCAN-main/DAppSCAN-source/contracts/PeckSh...,ReserveLogic.sol


In [7]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path])
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [8]:
# Roda a checagem
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    llm_categories.append(row['location'])  # Assuming 'bug_type' is the LLM's prediction
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")

    print("LOCATION: LINE - ", row['location'])
#print("LLM Categories:", llm_categories, "GT Categories:", gt_categories)



--- Rodando para idx 200 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/QuillAudits-1inch-token/1inch-token-99fd056f91005ca521a02a005f7bcd8f77e06afc/contracts/ERC20Permit.sol
LLM: Transaction Order Dependence (Race Condition)
GT: ['SWC-114-Transaction Order Dependence']

LOCATION: LINE -  53

--- Rodando para idx 202 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-GSPI Club-project2/openzeppelin-contracts-3.2.0/contracts/access/AccessControl.sol
LLM: Floating Pragma / Outdated Compiler Version
GT: ['SWC-103-Floating Pragma', 'SWC-102-Outdated Compiler Version']

LOCATION: LINE -  4

--- Rodando para idx 203 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/openzeppelin-EtherCamp_Hacker_Gold/virtual-accelerator-2529ffe5efd5294b44f1bc89dc9a4721a7b16409/HackerGold.sol
LLM: Integer Overflow/Underflow and Deprecated Throw Usage
GT: ['SWC-101-Integer Overflow and Underflow', 'SWC-135-Code With No Effects', 'SWC-135-Code With No Effects']

LOCATION: LINE -  Lines 30, 90

--

In [9]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")

100%|██████████| 194/194 [02:37<00:00,  1.23it/s]

Precision: 100.00
Recall: 67.01
F1-Score: 80.25
Confusion Matrix:
[[  0   0]
 [ 64 130]]


#### Zero-shot ToT Prompting

In [10]:
results_df = classify_dataset(subset, prompts.create_zeroshot_tot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

400it [1:16:32, 11.48s/it]

Precision: 58.93
Recall: 99.00
F1-Score: 73.88
Confusion Matrix:
[[ 62 138]
 [  2 198]]


,predicted_has_error,actual_has_error,bug_type,location
0,1,0,Unprotected fallback function allowing arbitra...,48-51
1,1,0,Arbitrary Storage Write Vulnerability,64-68
2,1,0,Multiple inheritance initialization conflict,94-95
3,1,0,Incorrect bitmask usage and missing access con...,26-34
4,1,0,Lack of access control allowing arbitrary stor...,"18-21, 28-31, 38-41, 48-51, 58-61, 68-71"


In [11]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

,predicted_has_error,actual_has_error,bug_type,location,file_path,file_name
200,1,1,NONE,NONE,DAppSCAN-main/DAppSCAN-source/contracts/QuillA...,ERC20Permit.sol
201,1,1,Floating Pragma,3,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
202,1,1,Floating Pragma and Outdated Compiler Version,4,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
203,1,1,Integer Overflow/Underflow,"30, 90",DAppSCAN-main/DAppSCAN-source/contracts/openze...,HackerGold.sol
204,1,1,Integer Underflow leading to logical error,121-124,DAppSCAN-main/DAppSCAN-source/contracts/PeckSh...,UserConfiguration.sol


In [12]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path])
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [13]:
# Roda a checagem
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    llm_categories.append(row['location'])  # Assuming 'bug_type' is the LLM's prediction
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")

    print("LOCATION: LINE - ", row['location'])
#print("LLM Categories:", llm_categories, "GT Categories:", gt_categories)



--- Rodando para idx 200 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/QuillAudits-1inch-token/1inch-token-99fd056f91005ca521a02a005f7bcd8f77e06afc/contracts/ERC20Permit.sol
LLM: NONE
GT: ['SWC-114-Transaction Order Dependence']

LOCATION: LINE -  NONE

--- Rodando para idx 201 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-SWAPP Protocol-project1/openzeppelin-contracts-3.3.0/contracts/access/AccessControl.sol
LLM: Floating Pragma
GT: ['SWC-103-Floating Pragma']

LOCATION: LINE -  3

--- Rodando para idx 202 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-GSPI Club-project2/openzeppelin-contracts-3.2.0/contracts/access/AccessControl.sol
LLM: Floating Pragma and Outdated Compiler Version
GT: ['SWC-103-Floating Pragma', 'SWC-102-Outdated Compiler Version']

LOCATION: LINE -  4

--- Rodando para idx 203 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/openzeppelin-EtherCamp_Hacker_Gold/virtual-accelerator-2529ffe5efd5294b44f1bc89dc9a4721a7b16409/Hacker

In [14]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")

100%|██████████| 198/198 [02:57<00:00,  1.11it/s]

Precision: 100.00
Recall: 54.55
F1-Score: 70.59
Confusion Matrix:
[[  0   0]
 [ 90 108]]
